# ⚖️ Bloque 3 — Factor de Carga y Rehashing
### Estructuras de Datos y Laboratorio — Universidad de Antioquia
**Curso:** Estructuras de Datos y Persistencia · Ingeniería de Sistemas  
**Unidad:** 2 — Hashing e índices basados en hash  
**Prerrequisito:** Bloque 2 — Función Hash y Colisiones  
**Prerrequisito para:** Hashing estático, Hashing dinámico (Clases 6 y 7)

---

## ¿Para qué sirve este notebook?

En el Bloque 2 se vio que las colisiones son inevitables y que el rendimiento de una tabla *hash* depende de cuántas claves comparten el mismo *bucket*. En este bloque se formaliza ese concepto con el ***load factor*** (factor de carga) α, se estudia cómo el rendimiento se degrada a medida que α crece, y se implementa el ***rehashing*** — el mecanismo que permite mantener α bajo control.

Al finalizar este notebook, se espera que pueda:

- Calcular e interpretar el factor de carga α de una tabla hash.
- Explicar cómo α afecta el costo de búsqueda, inserción y eliminación.
- Implementar el *rehashing* automático con la estrategia de duplicación.
- Calcular el número óptimo de *buckets* N dado un número esperado de claves.
- Reconocer los límites del hashing estático y la motivación para el hashing dinámico.

---

## Contenido

1. [Factor de carga (*Load Factor*)](#1-load-factor)
   - 1.1 Definición formal
   - 1.2 Impacto en el rendimiento
   - 1.3 Umbrales recomendados
2. [Rehashing](#2-rehashing)
   - 2.1 ¿Cuándo hacer rehashing?
   - 2.2 Estrategia de duplicación
   - 2.3 Costo amortizado del rehashing
3. [Tabla hash con rehashing automático](#3-tabla-con-rehashing)
4. [Cálculo del N óptimo](#4-n-optimo)
5. [Límites del hashing estático](#5-limites)
6. [Tabla de complejidad final](#6-complejidad)
7. [Ejercicios propuestos](#7-ejercicios)


---
## 1. Factor de carga (*Load Factor*)

### 1.1 Definición formal

El ***load factor*** (factor de carga) α es la relación entre el número de claves almacenadas `n` y el número de *buckets* `N` de la tabla:

$$\alpha = \frac{n}{N}$$

Donde:
- `n` = número de pares clave-valor actualmente en la tabla
- `N` = número total de *buckets* (capacidad de la tabla)
- α puede ser mayor que 1.0 en *separate chaining* (varios elementos por *bucket*)
- α siempre está en [0, 1) en *open addressing* (máximo un elemento por posición)

### 1.2 Impacto en el rendimiento

El factor de carga determina directamente el **número esperado de comparaciones** en una búsqueda:

| Estrategia | Costo de búsqueda esperado |
|---|---|
| *Separate chaining* (búsqueda exitosa) | `1 + α/2` |
| *Separate chaining* (búsqueda fallida) | `1 + α` |
| *Linear probing* (búsqueda exitosa) | `½ × (1 + 1/(1-α))` |
| *Linear probing* (búsqueda fallida) | `½ × (1 + 1/(1-α)²)` |

> 💡 **Observación clave:** Cuando α se acerca a 1.0 en *open addressing*, el término `1/(1-α)` tiende a infinito. El rendimiento colapsa mucho antes de que la tabla esté completamente llena.

### 1.3 Umbrales recomendados

| Estrategia | Umbral recomendado | ¿Qué pasa si se supera? |
|---|---|---|
| *Separate chaining* | α ≤ 0.75 | Listas largas, búsqueda se acerca a O(n) |
| *Linear probing* | α < 0.50 | *Clustering* severo, sondeos muy largos |
| *Quadratic probing* | α < 0.70 | *Clustering* secundario pronunciado |


In [ ]:
# Load factor impact: theoretical cost formulas

def chaining_cost_successful(alpha: float) -> float:
    """
    Expected number of comparisons for a SUCCESSFUL search
    in a separate chaining hash table.
    Formula: 1 + alpha/2
    """
    return 1 + alpha / 2


def chaining_cost_failed(alpha: float) -> float:
    """
    Expected number of comparisons for a FAILED search
    in a separate chaining hash table.
    Formula: 1 + alpha
    """
    return 1 + alpha


def linear_probing_cost_successful(alpha: float) -> float:
    """
    Expected number of probes for a SUCCESSFUL search
    in a linear probing hash table.
    Formula: 1/2 * (1 + 1/(1-alpha))
    """
    if alpha >= 1.0:
        return float('inf')
    return 0.5 * (1 + 1 / (1 - alpha))


def linear_probing_cost_failed(alpha: float) -> float:
    """
    Expected number of probes for a FAILED search
    in a linear probing hash table.
    Formula: 1/2 * (1 + 1/(1-alpha)^2)
    """
    if alpha >= 1.0:
        return float('inf')
    return 0.5 * (1 + 1 / (1 - alpha) ** 2)

#### Ejemplo 1 — Costo teórico según α

In [ ]:
# Example 1: Theoretical cost at different load factors
alphas = [0.1, 0.25, 0.50, 0.60, 0.70, 0.75, 0.80, 0.90, 0.95]

print(f"{'α':>6} | {'Chain (ok)':>12} | {'Chain (fail)':>14} | {'LinProbe (ok)':>15} | {'LinProbe (fail)':>17}")
print("-" * 75)

for a in alphas:
    cs  = chaining_cost_successful(a)
    cf  = chaining_cost_failed(a)
    lps = linear_probing_cost_successful(a)
    lpf = linear_probing_cost_failed(a)
    flag = " ⚠️" if a >= 0.75 else ""
    print(f"{a:>6.2f} | {cs:>12.2f} | {cf:>14.2f} | {lps:>15.2f} | {lpf:>17.2f}{flag}")

print()
print("⚠️  marks load factors above recommended thresholds.")

#### Ejemplo 2 — Visualizando la curva de degradación

In [ ]:
# Example 2: Plotting cost vs load factor using matplotlib
import matplotlib.pyplot as plt
import numpy as np

alphas_range = np.linspace(0.05, 0.95, 200)

chain_ok  = [chaining_cost_successful(a) for a in alphas_range]
chain_fail = [chaining_cost_failed(a) for a in alphas_range]
lp_ok     = [linear_probing_cost_successful(a) for a in alphas_range]
lp_fail   = [linear_probing_cost_failed(a) for a in alphas_range]

fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(alphas_range, chain_ok,   label='Chaining — successful search',   color='steelblue',  linewidth=2)
ax.plot(alphas_range, chain_fail, label='Chaining — failed search',        color='steelblue',  linewidth=2, linestyle='--')
ax.plot(alphas_range, lp_ok,      label='Linear probing — successful',     color='darkorange', linewidth=2)
ax.plot(alphas_range, lp_fail,    label='Linear probing — failed',         color='darkorange', linewidth=2, linestyle='--')

ax.axvline(x=0.75, color='red',   linestyle=':', linewidth=1.5, label='Chaining threshold (0.75)')
ax.axvline(x=0.50, color='green', linestyle=':', linewidth=1.5, label='Linear probing threshold (0.50)')

ax.set_xlim(0.05, 0.95)
ax.set_ylim(0, 12)
ax.set_xlabel('Load factor α', fontsize=12)
ax.set_ylabel('Expected comparisons / probes', fontsize=12)
ax.set_title('Performance degradation as load factor increases', fontsize=13)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 2. Rehashing

### 2.1 ¿Cuándo hacer rehashing?

El ***rehashing*** es el proceso de crear una tabla más grande y reinsertar todas las claves existentes en ella. Se activa cuando el factor de carga supera un umbral predefinido (típicamente 0.75 para *separate chaining*).

El proceso completo es:

```
1. Detectar: α = n/N > umbral
2. Crear nueva tabla con N' = siguiente primo después de 2×N
3. Para cada (key, value) en la tabla antigua:
       nueva_tabla.put(key, value)   ← h(k) cambia porque N cambió
4. Reemplazar la tabla antigua por la nueva
```

> ⚠️ **Punto crítico:** Al cambiar N, **todas las posiciones cambian**. Una clave que estaba en el *bucket* 3 con N=7 puede quedar en el *bucket* 10 con N=17. Por eso es obligatorio reinsertar **todas** las claves, no solo copiarlas.

### 2.2 Estrategia de duplicación

La estrategia más común es duplicar la capacidad y usar el siguiente número primo. Los números primos reducen el agrupamiento porque tienen menos factores comunes con las claves típicas.

```
N_nuevo = siguiente_primo(2 × N_actual)
```

### 2.3 Costo amortizado del rehashing

Aunque un *rehashing* individual cuesta O(n) — hay que reinsertar todas las claves — su **costo amortizado por inserción es O(1)**. La razón es la misma que en los *dynamic arrays*: el *rehashing* ocurre cada vez con menos frecuencia a medida que la tabla crece.

Si se duplica la tabla cada vez que se llena al 75%:
- Después del primer *rehash*: N → 2N, se puede insertar N/2 claves antes del siguiente
- Después del segundo: 2N → 4N, se puede insertar N claves antes del siguiente
- El costo total de todos los *rehashes* para n inserciones es O(n)
- Por lo tanto, el costo amortizado por inserción es **O(1)**


In [ ]:
# Helper: find the next prime number >= n

def is_prime(n: int) -> bool:
    """
    Check whether n is a prime number.

    Args:
        n (int): The number to check.

    Returns:
        bool: True if n is prime, False otherwise.
    """
    if n < 2:
        return False
    if n == 2:
        return True
    if n % 2 == 0:
        return False
    for i in range(3, int(n ** 0.5) + 1, 2):
        if n % i == 0:
            return False
    return True


def next_prime(n: int) -> int:
    """
    Return the smallest prime number >= n.

    Args:
        n (int): Starting point.

    Returns:
        int: The next prime >= n.
    """
    candidate = n if n >= 2 else 2
    while not is_prime(candidate):
        candidate += 1
    return candidate


# Quick test
for start in [7, 10, 14, 22, 50, 100]:
    print(f"  next_prime({start:3}) = {next_prime(start)}")

#### Ejemplo 3 — Observando el costo real del rehashing

In [ ]:
# Example 3: Tracking the cost of each rehash event

def simulate_rehashing(initial_n: int, n_insertions: int, threshold: float = 0.75):
    """
    Simulate insertions into a hash table with automatic rehashing.
    Tracks when rehash events occur and their cost.

    Args:
        initial_n    (int):   Initial number of buckets.
        n_insertions (int):   Total number of keys to insert.
        threshold    (float): Load factor threshold that triggers rehashing.
    """
    N = initial_n
    n = 0
    total_rehash_cost = 0
    rehash_events = []

    print(f"Initial N = {N}, threshold α = {threshold}")
    print(f"{'Insertion':>12} | {'n':>6} | {'N':>6} | {'α':>6} | {'Event':<30}")
    print("-" * 70)

    for i in range(1, n_insertions + 1):
        n += 1
        alpha = n / N

        if alpha > threshold:
            old_N = N
            N = next_prime(2 * N)
            total_rehash_cost += n  # Must reinsert all n keys
            rehash_events.append((i, old_N, N, n))
            alpha = n / N
            print(f"{i:>12} | {n:>6} | {N:>6} | {alpha:>6.2f} | *** REHASH: {old_N} → {N} (cost={n}) ***")
        elif i <= 5 or i % 20 == 0 or i == n_insertions:
            print(f"{i:>12} | {n:>6} | {N:>6} | {alpha:>6.2f} | normal insertion")

    print()
    print(f"Total rehash events     : {len(rehash_events)}")
    print(f"Total rehash cost (I/O) : {total_rehash_cost} reinsertions")
    print(f"Total insertions        : {n_insertions}")
    print(f"Amortized cost/insertion: {(n_insertions + total_rehash_cost) / n_insertions:.2f} ops")

simulate_rehashing(initial_n=7, n_insertions=100)

---
## 3. Tabla hash con rehashing automático

A continuación se presenta una implementación completa de una tabla *hash* con *separate chaining* que incorpora *rehashing* automático. Esta es la versión más robusta de las vistas en los Bloques 1 y 2.


#### Diagrama UML — `HashTableAutoRehash`

> 📌 **Espacio para diagrama UML**
>
> ```plantuml
> @startuml
> class HashTableAutoRehash {
>   - _buckets: list[list]
>   - _n_buckets: int
>   - _size: int
>   - _threshold: float
>   + __init__(initial_buckets: int, threshold: float)
>   + put(key: any, value: any) -> None
>   + get(key: any) -> any
>   + delete(key: any) -> bool
>   + load_factor() -> float
>   - _hash(key: any) -> int
>   - _rehash() -> None
>   + stats() -> None
>   + __str__() -> str
> }
> @enduml
> ```
>
> *(Renderice el diagrama y reemplace este bloque por la imagen generada)*


#### Documentación de la clase `HashTableAutoRehash`

| Elemento | Descripción |
|---|---|
| **Clase** | `HashTableAutoRehash` |
| **Propósito** | Tabla hash con *separate chaining* y *rehashing* automático al superar el umbral α |
| `_buckets` | Lista de listas; cada posición es un *bucket* con pares `(key, value)` |
| `_n_buckets` | Número actual de *buckets* — cambia con cada *rehash* |
| `_size` | Número de pares almacenados actualmente |
| `_threshold` | Umbral de factor de carga que dispara el *rehashing* (por defecto 0.75) |
| `put(key, value)` | Inserta o actualiza. Dispara *rehash* si α > umbral. **O(1)** amortizado |
| `get(key)` | Retorna el valor asociado. **O(1)** promedio |
| `delete(key)` | Elimina el par. **O(1)** promedio |
| `_rehash()` | Duplica la tabla y reinsertar todas las claves. **O(n)** — amortizado O(1) por inserción |
| `stats()` | Imprime métricas de la tabla: tamaño, *buckets*, α, distribución |


In [ ]:
# Hash Table with Automatic Rehashing (Separate Chaining)

class HashTableAutoRehash:
    """
    A production-ready hash table using separate chaining.
    Automatically rehashes when the load factor exceeds the threshold.

    This is the closest implementation to what real hash maps
    (like Python's dict) do internally.
    """

    def __init__(self, initial_buckets: int = 7, threshold: float = 0.75):
        """
        Initialize the hash table.

        Args:
            initial_buckets (int):   Initial number of buckets. Prefer a prime.
            threshold       (float): Load factor limit before rehashing.
        """
        self._n_buckets = next_prime(initial_buckets)
        self._buckets   = [[] for _ in range(self._n_buckets)]
        self._size      = 0
        self._threshold = threshold
        self._rehash_count = 0

    def _hash(self, key) -> int:
        """Internal hash function. O(1)"""
        return hash(key) % self._n_buckets

    def put(self, key, value) -> None:
        """
        Insert or update a key-value pair.
        Triggers rehashing automatically if α > threshold.
        Amortized O(1).

        Args:
            key:   The search key.
            value: The associated value.
        """
        index = self._hash(key)
        bucket = self._buckets[index]
        for i, (k, v) in enumerate(bucket):
            if k == key:
                bucket[i] = (key, value)
                return
        bucket.append((key, value))
        self._size += 1

        if self.load_factor() > self._threshold:
            self._rehash()

    def get(self, key):
        """
        Return the value for the given key. O(1) average.

        Raises:
            KeyError: If the key does not exist.
        """
        index = self._hash(key)
        for k, v in self._buckets[index]:
            if k == key:
                return v
        raise KeyError(f"Key '{key}' not found.")

    def delete(self, key) -> bool:
        """
        Remove the entry with the given key. O(1) average.

        Returns:
            True if removed, False if not found.
        """
        index = self._hash(key)
        bucket = self._buckets[index]
        for i, (k, v) in enumerate(bucket):
            if k == key:
                bucket.pop(i)
                self._size -= 1
                return True
        return False

    def _rehash(self) -> None:
        """
        Create a new table with roughly double the buckets
        and reinsert all existing key-value pairs.
        O(n) — but amortized O(1) per insertion.
        """
        old_buckets   = self._buckets
        old_n         = self._n_buckets
        self._n_buckets = next_prime(2 * old_n)
        self._buckets   = [[] for _ in range(self._n_buckets)]
        self._size      = 0
        self._rehash_count += 1

        print(f"  [rehash #{self._rehash_count}] N: {old_n} → {self._n_buckets}")

        for bucket in old_buckets:
            for key, value in bucket:
                self.put(key, value)

    def load_factor(self) -> float:
        """Return α = size / n_buckets."""
        return self._size / self._n_buckets

    def stats(self) -> None:
        """Print a summary of the table's internal state."""
        bucket_lengths = [len(b) for b in self._buckets]
        non_empty = sum(1 for l in bucket_lengths if l > 0)
        max_chain = max(bucket_lengths)
        print(f"  Buckets (N)      : {self._n_buckets}")
        print(f"  Stored keys (n)  : {self._size}")
        print(f"  Load factor (α)  : {self.load_factor():.3f}")
        print(f"  Non-empty buckets: {non_empty} / {self._n_buckets}")
        print(f"  Longest chain    : {max_chain}")
        print(f"  Rehash events    : {self._rehash_count}")

    def __str__(self) -> str:
        lines = []
        for i, bucket in enumerate(self._buckets):
            if bucket:
                lines.append(f"  [{i:3}]: {bucket}")
        return "\n".join(lines) if lines else "  (empty)"

#### Ejemplo 4 — Rehashing automático en acción

In [ ]:
# Example 4: Automatic rehashing triggered during insertions
import random
random.seed(7)

ht = HashTableAutoRehash(initial_buckets=7, threshold=0.75)

print("Inserting 50 random keys...")
print()
for i in range(50):
    key = random.randint(1, 500)
    ht.put(key, f"val_{key}")

print()
print("Final table statistics:")
ht.stats()

#### Ejemplo 5 — Comparando α antes y después del rehashing

In [ ]:
# Example 5: Tracking alpha before and after each rehash

class HashTableTracked(HashTableAutoRehash):
    """
    Extended version that logs alpha at every rehash event.
    """
    def _rehash(self):
        alpha_before = self.load_factor()
        old_n = self._n_buckets
        super()._rehash()
        alpha_after = self.load_factor()
        print(f"     α before: {alpha_before:.3f}  |  α after: {alpha_after:.3f}")

ht2 = HashTableTracked(initial_buckets=5, threshold=0.75)
print("Inserting 200 sequential keys (0 to 199):")
print()
for k in range(200):
    ht2.put(k, k * 10)

print()
print("Final statistics:")
ht2.stats()

---
## 4. Cálculo del N óptimo

### Repaso teórico

Cuando se conoce de antemano el número esperado de claves `n`, se puede calcular el número óptimo de *buckets* `N` directamente a partir del umbral α deseado:

$$N = \left\lceil \frac{n}{\alpha_{\text{objetivo}}} \right\rceil$$

Y luego se usa el siguiente primo mayor o igual a ese valor.

**Ejemplo del curso — Hashing estático:**  
Si se espera almacenar `n = 1,000,000` registros con α objetivo de 0.75 y páginas de disco que almacenan 630 pares por *bucket*:

```
N = ceil(1,000,000 / (0.75 × 630)) ≈ 2,117 buckets
```

Este es exactamente el cálculo que se usa en la **Clase 6 — Hashing estático** del curso.


In [ ]:
# Optimal N calculation for static hashing
import math

def optimal_buckets(n_keys: int,
                    target_alpha: float = 0.75,
                    records_per_bucket: int = 1) -> dict:
    """
    Calculate the optimal number of buckets for a hash table.

    Args:
        n_keys             (int):   Expected number of keys.
        target_alpha       (float): Desired load factor (default 0.75).
        records_per_bucket (int):   Records per bucket page (for disk-based hashing).

    Returns:
        dict: Contains raw N, next prime N, and achieved alpha.
    """
    raw_N    = math.ceil(n_keys / (target_alpha * records_per_bucket))
    prime_N  = next_prime(raw_N)
    achieved = n_keys / (prime_N * records_per_bucket)
    return {
        'n_keys'            : n_keys,
        'target_alpha'      : target_alpha,
        'records_per_bucket': records_per_bucket,
        'raw_N'             : raw_N,
        'prime_N'           : prime_N,
        'achieved_alpha'    : achieved
    }


# In-memory hash table
result_mem = optimal_buckets(n_keys=10_000, target_alpha=0.75)
print("In-memory hash table:")
for k, v in result_mem.items():
    print(f"  {k:<22}: {v}")

print()

# Disk-based static hashing (Class 6 example)
result_disk = optimal_buckets(n_keys=1_000_000, target_alpha=0.75, records_per_bucket=630)
print("Disk-based static hashing (Class 6 scenario):")
for k, v in result_disk.items():
    print(f"  {k:<22}: {v:.4f}" if isinstance(v, float) else f"  {k:<22}: {v}")

#### Ejemplo 6 — Sensibilidad del rendimiento al N elegido

In [ ]:
# Example 6: How the choice of N affects performance for a fixed number of keys
n_keys = 1000

print(f"Fixed number of keys: {n_keys}")
print()
print(f"{'N buckets':>12} | {'α':>6} | {'Chain cost (ok)':>18} | {'LinProbe cost (ok)':>20} | {'Assessment':<20}")
print("-" * 85)

for N in [500, 750, 1000, 1333, 2000, 5000]:
    alpha = n_keys / N
    chain = chaining_cost_successful(alpha) if alpha <= 2.0 else float('inf')
    lp    = linear_probing_cost_successful(alpha) if alpha < 1.0 else float('inf')
    if alpha < 0.5:
        note = "memory-wasteful"
    elif alpha <= 0.75:
        note = "✓ optimal zone"
    elif alpha <= 1.0:
        note = "⚠️  degraded"
    else:
        note = "✗ overloaded"
    lp_str = f"{lp:>20.2f}" if lp != float('inf') else f"{'∞':>20}"
    print(f"{N:>12} | {alpha:>6.2f} | {chain:>18.2f} | {lp_str} | {note:<20}")

---
## 5. Límites del hashing estático

### Repaso teórico

El *rehashing* resuelve el problema del factor de carga en memoria, pero en el contexto de **bases de datos orientadas a disco** introduce un problema mucho más grave:

**Costo del rehashing en disco:**
- En memoria: copiar n elementos cuesta O(n) operaciones de CPU — rápido.
- En disco: leer y escribir n páginas cuesta **O(n) operaciones de I/O** — extremadamente lento.

Si la tabla tiene 2,117 *buckets* y cada *bucket* es una página de disco, un *rehashing* requiere leer y escribir **más de 4,000 páginas**. Con un costo promedio de D = 10ms por página, eso son **40 segundos de inactividad total**.

Esta es la motivación fundamental para el **hashing dinámico** (Clase 7): estrategias que crecen **incrementalmente**, una pequeña parte a la vez, sin necesidad de reconstruir toda la tabla.

> 💡 **Resumen de los límites del hashing estático:**
> 1. N es fijo: si se insertan más claves de las previstas, α sube sin control.
> 2. El *rehashing* en disco es prohibitivamente costoso.
> 3. No soporta búsquedas por rango (*range queries*) — solo búsquedas exactas.
> 4. Requiere conocer de antemano el número aproximado de claves.


In [ ]:
# Disk rehashing cost model

def disk_rehash_cost(n_buckets: int,
                     page_read_ms: float = 10.0,
                     page_write_ms: float = 10.0) -> dict:
    """
    Estimate the I/O cost of a rehash operation for a disk-based hash table.

    Assumptions:
      - Every bucket is one disk page.
      - Rehashing doubles the table: reads n_buckets pages, writes 2*n_buckets pages.

    Args:
        n_buckets      (int):   Current number of bucket pages.
        page_read_ms   (float): Time to read one page (ms).
        page_write_ms  (float): Time to write one page (ms).

    Returns:
        dict: Breakdown of I/O operations and total time.
    """
    reads       = n_buckets
    writes      = 2 * n_buckets
    total_io    = reads + writes
    total_ms    = reads * page_read_ms + writes * page_write_ms
    return {
        'current_buckets' : n_buckets,
        'new_buckets'     : 2 * n_buckets,
        'pages_read'      : reads,
        'pages_written'   : writes,
        'total_io_ops'    : total_io,
        'total_time_ms'   : total_ms,
        'total_time_sec'  : total_ms / 1000
    }


# Class 6 scenario: static hash table with 2117 buckets
result = disk_rehash_cost(n_buckets=2117)
print("Disk rehash cost — Class 6 scenario (N=2117 bucket pages):")
print()
for k, v in result.items():
    print(f"  {k:<22}: {v:,.1f}" if isinstance(v, float) else f"  {k:<22}: {v:,}")

print()
print("This is why static hashing on disk is replaced by dynamic hashing (Class 7).")

---
## 6. Tabla de complejidad final

### Resumen consolidado — Notación O

Esta tabla consolida el costo de las operaciones de los tres bloques de repaso y sirve como referencia directa para las Clases 6 y 7.

| Operación | Separate Chaining | Open Addressing | Hashing estático (disco) |
|---|:---:|:---:|:---:|
| Búsqueda exacta (promedio) | O(1 + α) | O(1/(1-α)) | **O(1) — 2 I/Os** |
| Búsqueda exacta (peor caso) | O(n) | O(n) | O(overflow chain) |
| Búsqueda por rango | ✗ No soportada | ✗ No soportada | ✗ No soportada |
| Inserción (promedio) | O(1) amort. | O(1/(1-α)) | O(1) si hay espacio |
| Rehashing (en memoria) | O(n) — amort. O(1)/ins. | O(n) — amort. O(1)/ins. | — |
| Rehashing (en disco) | — | — | **O(n) I/Os — prohibitivo** |
| Factor de carga recomendado | α ≤ 0.75 | α < 0.70 | α ≈ 0.75 (calculado al crear) |

### Mapa hacia los siguientes temas

```
Bloque 3 (este notebook)
       ↓
  Clase 6 — Hashing estático: N fijo, cálculo de N óptimo, overflow chaining
       ↓
  Clase 7 — Hashing dinámico: crecimiento incremental sin rehashing global
            ├── Hashing lineal (Linear Hashing)
            └── Hashing extensible (Extendible Hashing)
```


---
## 7. Ejercicios propuestos

Se recomienda resolver los ejercicios antes de las Clases 6 y 7.

In [ ]:
# Exercise 1
# A hash table starts with N=11 buckets and threshold α=0.75.
# Keys to insert: [5, 3, 8, 10, 9, 7, 14, 21, 36]
#
# a) At which insertion does the first rehash trigger?
# b) What is the new N after the rehash? (use next prime after 2×11)
# c) Which keys change their bucket index after the rehash?
#    Show the old bucket (key % 11) and new bucket (key % new_N) for each key.

# Write your solution here


In [ ]:
# Exercise 2
# A disk-based static hash table must store n=500,000 records.
# Each bucket page holds up to 400 records. Target α = 0.80.
#
# a) Calculate the optimal number of bucket pages N.
# b) What is the achieved load factor with the next prime N?
# c) If page I/O costs 8ms, how long would a full rehash take?
# d) At what load factor should an alert be raised to warn
#    the administrator before performance degrades too much?

# Write your solution here


In [ ]:
# Exercise 3 (challenge)
# Implement a ShrinkableHashTable that extends HashTableAutoRehash with:
#
# - A lower threshold (e.g. α < 0.25) that triggers a SHRINK operation:
#   halves the number of buckets (using the previous prime below N/2)
#   and reinserts all keys into the smaller table.
#
# - Override put() and delete() to check both thresholds.
#
# Test your implementation by:
#   1. Inserting 100 keys (should trigger several grow rehashes)
#   2. Deleting 80 keys  (should trigger at least one shrink)
#   3. Printing stats() before and after each phase.
#
# Why is shrinking less common in practice than growing?

# Write your solution here


## Referencias de este bloque

### Referencias principales

- **Goodrich, Tamassia & Goldwasser (2013).** *Data Structures and Algorithms in Python*. Wiley. **Capítulo 10** — secciones sobre load factor y rehashing.
- **Ramakrishnan & Gehrke (2003).** *Database Management Systems* (3rd ed.). McGraw-Hill. **Capítulo de Hashing** — cálculo de N óptimo para hashing estático en disco.
- **Bhargava, A. (2016).** *Grokking Algorithms*. Manning. **Capítulo 5** — intuición sobre el factor de carga.
- Visualizador interactivo: [USFCA — Open Hash](https://www.cs.usfca.edu/~galles/visualization/OpenHash.html)
- Visualizador interactivo: [USFCA — Closed Hash](https://www.cs.usfca.edu/~galles/visualization/ClosedHash.html)

---

### Conexión con Silberschatz — *Database System Concepts* (7th ed.)

| Sección | Tema | Conecta con |
|---|---|---|
| Cap. 14 — §14.4.1 | Hash Indexes | Factor de carga α como criterio de diseño del índice |
| Cap. 14 — §14.4.3 | Cálculo de N | N óptimo de *buckets* para minimizar *overflow* |
| Cap. 14 — §14.5 | Dynamic Hashing | Limitaciones del *rehashing* en disco → motivación |
| Cap. 14 — §14.5.1 | Extendible Hashing | Directamente la Clase 7 del curso |

<br>

> 💡 **Lectura recomendada:** leer §14.4.3 y §14.5 después de completar este notebook. Es la lectura puente directa hacia las Clases 6 y 7.